# Tasks 6 & 7: Feature Selection and Predictive Model

This notebook defines the features for predicting total piece travel time, trains an XGBoost Regressor, and evaluates its performance.

### The prediction problem

**Goal**: Predict the total travel time to the quench bath (`lifetime_bath_s`) using only data available **early in the process** — before the piece has finished the line.

**Why this is useful**: If we can predict the total time after only the 2nd strike, we can raise real-time alerts for pieces that are likely to be slow, allowing operators to investigate the cause while the piece is still on the line.

### Feature selection rationale

**Constraint**: the model must predict using only information available **after the 2nd strike** — approximately 18 seconds into the ~58-second journey. Any feature that requires waiting for later stages cannot be used, because by the time those values exist, the prediction is no longer useful.

#### Selected features

| Feature | Type | Available at | Why include it |
|---|---|---|---|
| `die_matrix` | Categorical (int) | Before processing | Each die has different tooling geometry and expected times — matrix 4974 has a median bath time of ~56s while 5091 has ~59s |
| `lifetime_2nd_strike_s` | Continuous (seconds) | After 2nd strike (~18s) | The earliest cumulative time — if the piece is already slow here, it will likely carry that delay through to the bath |
| `oee_cycle_time_s` | Continuous (seconds) | Rolling metric | Production rate context — slower OEE may correlate with systematic delays (robot trajectory adjustments, hydraulic pressure drops) |

#### Excluded features

| Feature | Why excluded |
|---|---|
| `lifetime_3rd_strike_s` | Available too late (~25s) — the piece is already halfway through the main press |
| `lifetime_4th_strike_s` | Available too late (~38s) — and has ~16% missing data from a sensor offline period |
| `lifetime_auxiliary_press_s` | Available too late (~55s) — only ~2s before the bath, prediction would be useless |
| `lifetime_general_s` | Equivalent to `lifetime_bath_s` — redundant with the target |
| `partial_*` columns | Derived from cumulative times that include late-stage data |
| `piece_id` | Not predictive — just an identifier |

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

GOLD_PATH = Path("../data/gold/pieces.parquet")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

print("Libraries loaded ✓")

Libraries loaded ✓


## 1. Load gold dataset

In [2]:
gold = pd.read_parquet(GOLD_PATH)
print(f"Gold dataset: {len(gold):,} pieces")
print(f"Die matrices: {sorted(gold['die_matrix'].unique())}")
display(gold.head(3))

Gold dataset: 169,161 pieces
Die matrices: [np.int64(4974), np.int64(5052), np.int64(5090), np.int64(5091)]


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s,partial_furnace_to_2nd_strike_s,partial_2nd_to_3rd_strike_s,partial_3rd_to_4th_strike_s,partial_4th_strike_to_auxiliary_press_s,partial_auxiliary_press_to_bath_s,oee_cycle_time_s,after_gap,production_run_id
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,54.599998,56.200001,56.200001,17.900000,6.700001,13.400000,16.599998,1.600002,NaN,True,1
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,54.799999,56.400002,56.400002,17.900000,6.700001,13.300001,16.899998,1.600002,NaN,False,1
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,55.299999,56.900002,56.900002,18.200001,6.599998,13.500000,17.000000,1.600002,NaN,False,1


## 2. Prepare features and target

- **Features (X)**: `die_matrix`, `lifetime_2nd_strike_s`, `oee_cycle_time_s`
- **Target (y)**: `lifetime_bath_s`

Drop rows where any feature or the target is NULL (missing drill data does not affect us here since we don't use drill as a feature).

In [3]:
# Prepare features and target
FEATURES = ['die_matrix', 'lifetime_2nd_strike_s', 'oee_cycle_time_s']
TARGET = 'lifetime_bath_s'

# Handle missing OEE: fill with median (~13.8s) as per model design
oee_median = gold['oee_cycle_time_s'].median()
print(f"OEE median (for imputation): {oee_median:.1f}s")
print(f"OEE null count: {gold['oee_cycle_time_s'].isna().sum():,} ({gold['oee_cycle_time_s'].isna().mean()*100:.1f}%)")

df = gold[FEATURES + [TARGET]].copy()
df['oee_cycle_time_s'] = df['oee_cycle_time_s'].fillna(oee_median)

# Drop any remaining nulls (e.g., missing bath time)
before = len(df)
df = df.dropna()
print(f"\nPieces with all features + target: {len(df):,} (dropped {before - len(df):,})")

X = df[FEATURES]
y = df[TARGET]

print(f"\nFeatures (X): {list(X.columns)}")
print(f"Target (y): {TARGET}")
print(f"Shape: X={X.shape}, y={y.shape}")

OEE median (for imputation): 13.8s
OEE null count: 38,095 (22.5%)

Pieces with all features + target: 167,736 (dropped 1,425)

Features (X): ['die_matrix', 'lifetime_2nd_strike_s', 'oee_cycle_time_s']
Target (y): lifetime_bath_s
Shape: X=(167736, 3), y=(167736,)


## 3. Feature correlation with target

How strongly does each feature correlate with the total bath time? High correlation suggests predictive value.

In [4]:
# Pearson correlation between features and target
correlations = df[FEATURES + [TARGET]].corr()[TARGET].drop(TARGET)
print("Feature correlations with lifetime_bath_s:")
for feat, corr in correlations.items():
    strength = "strong" if abs(corr) > 0.5 else "moderate" if abs(corr) > 0.3 else "weak"
    print(f"  {feat:25s}: {corr:+.4f} ({strength})")

Feature correlations with lifetime_bath_s:
  die_matrix               : +0.2266 (weak)
  lifetime_2nd_strike_s    : +0.7660 (strong)
  oee_cycle_time_s         : +0.3434 (moderate)


## 4. Train/test split

Split 80/20 with a fixed random seed for reproducibility. Stratify by die_matrix to ensure each matrix is represented proportionally in both sets.

In [5]:
# 80/20 split stratified by die_matrix
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=X['die_matrix']
)

print(f"Training set: {len(X_train):,} pieces")
print(f"Test set:     {len(X_test):,} pieces")
print(f"\nPer-matrix distribution:")
for matrix in sorted(X['die_matrix'].unique()):
    n_train = (X_train['die_matrix'] == matrix).sum()
    n_test = (X_test['die_matrix'] == matrix).sum()
    print(f"  Matrix {matrix}: train={n_train:,}, test={n_test:,}")

Training set: 134,188 pieces
Test set:     33,548 pieces

Per-matrix distribution:
  Matrix 4974: train=12,426, test=3,106
  Matrix 5052: train=16,721, test=4,181
  Matrix 5090: train=65,360, test=16,341
  Matrix 5091: train=39,681, test=9,920


## 5. Train XGBoost Regressor

XGBoost is chosen because:
- Handles mixed feature types (categorical die_matrix + continuous times)
- Robust to the remaining noise in the data
- Fast training on ~100k rows
- Produces feature importance rankings

In [6]:
# Train XGBoost Regressor
model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)
print(f"Model trained: {model.n_estimators} trees, depth {model.max_depth}")

Model trained: 200 trees, depth 6


## 6. Evaluate on test set

Key metrics:
- **RMSE**: root mean squared error (same unit as target — seconds)
- **MAE**: mean absolute error (average prediction error in seconds)
- **R²**: coefficient of determination (1.0 = perfect, 0.0 = no better than mean)

In [7]:
# Evaluate on test set
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Test Set Performance:")
print(f"  RMSE: {rmse:.3f}s")
print(f"  MAE:  {mae:.3f}s")
print(f"  R²:   {r2:.4f}")
print(f"\nContext: on a ~58s bath time, MAE of {mae:.2f}s = ±{mae/58*100:.1f}% error")

Test Set Performance:
  RMSE: 1.837s
  MAE:  0.918s
  R²:   0.6873

Context: on a ~58s bath time, MAE of 0.92s = ±1.6% error


## 7. Performance per die matrix

Check if the model performs equally well across all matrices, or if some are harder to predict.

In [8]:
# Per-matrix performance
print("Performance per die matrix:")
print(f"{'Matrix':>8s} {'RMSE':>8s} {'MAE':>8s} {'R²':>8s} {'Pieces':>8s}")
print("-" * 45)
for matrix in sorted(X_test['die_matrix'].unique()):
    mask = X_test['die_matrix'] == matrix
    y_t = y_test[mask]
    y_p = y_pred[mask]
    m_rmse = np.sqrt(mean_squared_error(y_t, y_p))
    m_mae = mean_absolute_error(y_t, y_p)
    m_r2 = r2_score(y_t, y_p)
    print(f"{int(matrix):>8d} {m_rmse:>8.3f} {m_mae:>8.3f} {m_r2:>8.4f} {mask.sum():>8,}")

Performance per die matrix:
  Matrix     RMSE      MAE       R²   Pieces
---------------------------------------------
    4974    0.932    0.470   0.7271    3,106
    5052    1.459    0.706   0.6976    4,181
    5090    2.037    1.107   0.6454   16,341
    5091    1.846    0.838   0.6914    9,920


## 8. Feature importance

Which features contribute most to the prediction? This validates the feature selection rationale.

In [9]:
# Feature importance (gain-based)
importance = model.get_booster().get_score(importance_type='gain')
total_gain = sum(importance.values())
print("Feature importance (gain-based):")
for feat, gain in sorted(importance.items(), key=lambda x: -x[1]):
    pct = gain / total_gain * 100
    print(f"  {feat:25s}: {gain:10.1f} ({pct:.1f}%)")

Feature importance (gain-based):
  lifetime_2nd_strike_s    :      761.3 (71.3%)
  oee_cycle_time_s         :      224.2 (21.0%)
  die_matrix               :       82.1 (7.7%)


## 9. Save model and metadata

Save the trained model and its metadata for use by the inference service (Task 8).

In [10]:
# Save model and metadata
model_path = MODEL_DIR / "xgboost_bath_predictor.json"
model.save_model(str(model_path))
print(f"Model saved to {model_path}")

metadata = {
    "model_file": "xgboost_bath_predictor.json",
    "features": FEATURES,
    "target": TARGET,
    "metrics": {
        "rmse": round(rmse, 3),
        "mae": round(mae, 3),
        "r2": round(r2, 4)
    },
    "training": {
        "n_samples_train": len(X_train),
        "n_samples_test": len(X_test),
        "random_state": 42,
        "hyperparameters": {
            "n_estimators": 200,
            "max_depth": 6,
            "learning_rate": 0.1,
            "subsample": 0.8,
            "colsample_bytree": 0.8
        }
    },
    "die_matrices": sorted(int(m) for m in X['die_matrix'].unique()),
    "oee_median": round(oee_median, 1)
}

metadata_path = MODEL_DIR / "model_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved to {metadata_path}")

Model saved to ../models/xgboost_bath_predictor.json
Metadata saved to ../models/model_metadata.json


## 10. Summary

In [11]:
# Summary
print("=" * 60)
print("MODEL TRAINING SUMMARY")
print("=" * 60)
print(f"""
Model: XGBoost Regressor (200 trees, depth 6)
Target: lifetime_bath_s (~58s total travel time)
Features: die_matrix, lifetime_2nd_strike_s, oee_cycle_time_s

Performance (test set):
  RMSE: {rmse:.3f}s
  MAE:  {mae:.3f}s (±{mae/58*100:.1f}% on ~58s)
  R²:   {r2:.4f}

Saved artifacts:
  models/xgboost_bath_predictor.json
  models/model_metadata.json
""")

MODEL TRAINING SUMMARY

Model: XGBoost Regressor (200 trees, depth 6)
Target: lifetime_bath_s (~58s total travel time)
Features: die_matrix, lifetime_2nd_strike_s, oee_cycle_time_s

Performance (test set):
  RMSE: 1.837s
  MAE:  0.918s (±1.6% on ~58s)
  R²:   0.6873

Saved artifacts:
  models/xgboost_bath_predictor.json
  models/model_metadata.json

